In [185]:
text = "<s>I love NLP</s><s>I love pizza</s><s>NLP is fun</s><s>Python is great</s><s>I love programming</s><s>Python is awesome</s><s>I love coding</s><s>NLP is amazing</s><s>Python is powerful</s><s>I love AI</s><s>NLP love tokenization</s><s>Python is versatile</s><s>I love machine learning</s><s>NLP is fascinating</s><s>Python is popular</s><s>I love deep learning</s><s>NLP is evolving</s><s>Python is efficient</s><s>I love data science</s><s>NLP is exciting</s>"

In [186]:
import re

result = re.split(r"<s>|\s+|(</s>)", text)
result = [x for x in result if x]

print(result)

['I', 'love', 'NLP', '</s>', 'I', 'love', 'pizza', '</s>', 'NLP', 'is', 'fun', '</s>', 'Python', 'is', 'great', '</s>', 'I', 'love', 'programming', '</s>', 'Python', 'is', 'awesome', '</s>', 'I', 'love', 'coding', '</s>', 'NLP', 'is', 'amazing', '</s>', 'Python', 'is', 'powerful', '</s>', 'I', 'love', 'AI', '</s>', 'NLP', 'love', 'tokenization', '</s>', 'Python', 'is', 'versatile', '</s>', 'I', 'love', 'machine', 'learning', '</s>', 'NLP', 'is', 'fascinating', '</s>', 'Python', 'is', 'popular', '</s>', 'I', 'love', 'deep', 'learning', '</s>', 'NLP', 'is', 'evolving', '</s>', 'Python', 'is', 'efficient', '</s>', 'I', 'love', 'data', 'science', '</s>', 'NLP', 'is', 'exciting', '</s>']


In [187]:
from collections import Counter
word_counts = Counter(result)
word_counts = dict(word_counts)
total_words = sum(word_counts.values())
total_words = float(total_words)
for word in word_counts:
    word_counts[word] /= total_words

In [188]:
word_counts

{'I': 0.0963855421686747,
 'love': 0.10843373493975904,
 'NLP': 0.08433734939759036,
 '</s>': 0.24096385542168675,
 'pizza': 0.012048192771084338,
 'is': 0.13253012048192772,
 'fun': 0.012048192771084338,
 'Python': 0.07228915662650602,
 'great': 0.012048192771084338,
 'programming': 0.012048192771084338,
 'awesome': 0.012048192771084338,
 'coding': 0.012048192771084338,
 'amazing': 0.012048192771084338,
 'powerful': 0.012048192771084338,
 'AI': 0.012048192771084338,
 'tokenization': 0.012048192771084338,
 'versatile': 0.012048192771084338,
 'machine': 0.012048192771084338,
 'learning': 0.024096385542168676,
 'fascinating': 0.012048192771084338,
 'popular': 0.012048192771084338,
 'deep': 0.012048192771084338,
 'evolving': 0.012048192771084338,
 'efficient': 0.012048192771084338,
 'data': 0.012048192771084338,
 'science': 0.012048192771084338,
 'exciting': 0.012048192771084338}

In [189]:
# Total sum of probabilities rule
s = 0.0
for word in word_counts:
    s += word_counts[word]
s

1.0000000000000007

In [190]:
def P_single(word):
    if word in word_counts:
        return word_counts[word]
    else:
        return 0.0

In [191]:
P_single("love")

0.10843373493975904

In [192]:
# P_join function is count(word1, word2, word3, ...) / total_words
def P_joint(*args):
    count = 0
    for i in range(len(result) - len(args) + 1):
        if tuple(result[i:i + len(args)]) == args:
            count += 1
    return count / total_words

In [193]:
P_joint("love", "NLP")

0.012048192771084338

In [194]:
# P_conditional function is P(wordn | wordn-1, wordn-2, ..., word1) = P(word1, word2, ..., wordn) / P(word1, word2, ..., wordn-1)
def P_conditional(target, *given):
    if not given:
        raise ValueError("Need at least one conditioning word.")
    denom = P_joint(*given)
    return P_joint(*given, target) / denom if denom else 0.0

In [195]:
print(f" P(NLP | love) = {P_conditional('NLP', 'love')}")
print(f" P(love | I) = {P_conditional('love', 'I')}")

 P(NLP | love) = 0.11111111111111112
 P(love | I) = 1.0


In [196]:
P_joint("I", "love")

0.0963855421686747

In [197]:
# Chain rule: P(A, B, C) = P(A) * P(B | A) * P(C | A, B)
text = "NLP love pizza</s>"
# P(NLP) * P(love | NLP) * P(pizza | NLP, love)
print(f"P(NLP) = {P_single('NLP')}")
print(f"P(love | NLP) = {P_conditional('love', 'NLP')}")
print(f"P(pizza | NLP, love) = {P_conditional('pizza', 'NLP', 'love')}")
print(f"P(</s> | NLP, love, pizza) = {P_conditional('</s>', 'NLP', 'love', 'pizza')}")

# Calculate the probability of the entire text using the chain rule
def P_chain_rule(text):
    words = re.split(r"<s>|\s+|(</s>)", text)
    p = P_single(words[0])
    for i in range(1, len(words)):
        p *= P_conditional(words[i], *words[:i])
    return p

print(f"P(NLP love pizza</s>) = {P_chain_rule(text)}")

P(NLP) = 0.08433734939759036
P(love | NLP) = 0.14285714285714288
P(pizza | NLP, love) = 0.0
P(</s> | NLP, love, pizza) = 0.0
P(NLP love pizza</s>) = 0.0


In [198]:
# Law of total probability: P(A) = P(A | B0) * P(B0) + P(A | B1) * P(B1) + ... + P(A | Bn) * P(Bn)
def P_total_probability(A, B_list):
    total_prob = 0.0
    for B in B_list:
        total_prob += P_conditional(A, B) * P_single(B)
    return total_prob

In [199]:
P_total_probability("love", ["I", "NLP", "Python"])

0.10843373493975904

In [200]:
def bayes_theorem(A, B):
    P_A_given_B = P_conditional(A, B)
    P_B = P_single(B)
    P_A = P_single(A)
    
    if P_B == 0:
        return 0.0
    
    return (P_A_given_B * P_B) / P_A  # P_B_given_A = (P_A_given_B * P_B) / P_A

In [201]:
P_nlp_given_love = P_conditional("NLP", "love")
P_love_given_nlp = bayes_theorem("NLP", "love")

print(f"P(NLP | love) = {P_nlp_given_love}")
print(f"P(love | NLP) = {P_love_given_nlp}")

P(NLP | love) = 0.11111111111111112
P(love | NLP) = 0.14285714285714288
